# ModelArts Notebook 连接 MRS Hive（Kerberos 安全集群）

在本 notebook（与 MRS 同 VPC、可出公网）中完成：**环境预装 → 网络探测 → krb5 配置 → kinit → Kerberos 连接 HiveServer2 → 验证查询 → 通用查询工具**。

> ✅ **2026-08-19 已在真实 ModelArts Notebook 实测通过**（CPU 镜像 / ma-user 无 root）：第 7 格输出 `databases: ['default', 'mrs_system']`、`breast_cancer 行数: 569`。

配方来自 2026-08-19 在 master1 节点上的实测排障，事实的唯一权威来源是 `hive_export/MRS_RUN.md §0`，决策记录见 `docs/adr/0002`：

| 关键事实 | 值 |
|---|---|
| HiveServer2 | `10.0.0.15:21066`（内网 IP，binary transport） |
| SPN 中间段 | `hadoop.252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com`（`hadoop.` + 域名全小写；`haddop_` 变体在 KDC 里**不存在**） |
| KDC 端口 | **21732**（不是开源默认的 88；master1=10.0.0.15、master2=10.0.0.51） |
| SASL QOP | **auth-conf**（加密包装层）→ 必须用 cyrus 的 `sasl` pip 包；pure-sasl+pykerberos 实测在加密包装阶段失败 |

**前置条件**：
1. 安全组放行 notebook → `10.0.0.15:21066` 与 `10.0.0.15/10.0.0.51:21732`（TCP+UDP）；
2. notebook 能访问公网（pip 装依赖）；
3. root / 免密 sudo / 均无（ma-user）三种环境均可 —— 第 3 格自动适配。

> kinit 凭据：本 notebook 用 `getpass` 交互输入 hhx 密码（不在代码留明文）；免交互的 keytab 方式见第 8 节末尾。
>
> _English edition: `modelarts_hive_conn_EN.ipynb`（与中文版同提交同步，见 ADR-0001）。_

In [ ]:
# ================== 1. 连接配置（实测值，换集群时改这里） ==================
HIVE_HOST = "10.0.0.15"    # HiveServer2 内网 IP（master1）
HIVE_PORT = 21066          # HiveServer2 Thrift 端口
DATABASE  = "default"
USERNAME  = "hhx"          # MRS 业务用户

REALM    = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"  # MRS 系统域名(Realm)
SPN_HOST = "hadoop." + REALM.lower()   # 实测正确的 SPN 中间段（haddop_ 变体是错的）

KDC_HOSTS = ["10.0.0.15", "10.0.0.51"]  # 两个 Master 都写，容错
KDC_PORT  = 21732                        # 华为 MRS 专用 KDC 端口，不是 88！

print("principal =", f"hive/{SPN_HOST}@{REALM}")

In [ ]:
# ================== 2. 网络探测（安全组没放行在这里快速暴露） ==================
import socket

def probe(host, port, name, timeout=5):
    s = socket.socket(); s.settimeout(timeout)
    try:
        s.connect((host, port)); print(f"[OK]   {name} {host}:{port} 可达")
        return True
    except Exception as e:
        print(f"[FAIL] {name} {host}:{port} 不可达: {e}")
        return False
    finally:
        s.close()

net_ok = probe(HIVE_HOST, HIVE_PORT, "HiveServer2")
for k in KDC_HOSTS:
    net_ok &= probe(k, KDC_PORT, "KDC")

assert net_ok, (
    "网络不通：请确认 notebook 与 MRS 同 VPC，且安全组放行 21066 与 21732(TCP+UDP)。\n"
    "只通 21066 不够 —— kinit 还要访问 KDC 的 21732。"
)

In [ ]:
# ================== 3. 环境预装（幂等；自动适配三种环境；实时进度） ==================
# 目标产物: kinit 二进制 + cyrus 的 sasl(GSSAPI 插件在位) + 纯 python 的 pyhive 等
#   ★ 集群 qop=auth-conf，必须用 cyrus 的 sasl；pure-sasl+pykerberos 实测在
#     加密包装阶段报 "Invalid token was supplied"（见 ADR-0002）
# 适配顺序（打印 [env] 说明命中哪支）：
#   A. root            -> apt 装 gcc/g++/krb5-user/头文件 + GSSAPI 插件, pip 编译 sasl
#   B. ma-user+免密sudo -> 同 A，apt 前加 sudo -n
#   C. 无 root(常见)   -> conda-forge 预编译: krb5(自带 kinit) + sasl，无需编译器
import collections, importlib, os, re, shutil, subprocess, sys, threading, time

def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

# ---- 实时进度执行器: 关键行即时打印(带耗时), 静默期打心跳, 失败回放末尾输出 ----
_BAR = re.compile(r"^\W*\[\W*\d+%\W*\]\W*$")            # apt 的 [ 12%] 进度条(噪声)
_HOT = re.compile(r"solving|collecting|downloading|extracting|preparing|executing|"
                  r"transaction|unpacking|setting up|processing|fetched|^get|^hit|"
                  r"building wheel|successfully|installed|nothing to do|all requested|"
                  r"error|fail|conflict|warn", re.I)     # 值得展示的进度/结果行

def run_stream(cmd, note=None, heartbeat=20):
    """流式执行外部命令。返回码 0=成功; 失败时自动回放末尾 40 行输出。"""
    if note: print(f"[run] {note}", flush=True)
    t0, tail = time.time(), collections.deque(maxlen=40)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    stop = threading.Event()
    def _beat():                                          # 长时间静默时证明"还活着"
        quiet = time.time()
        while not stop.wait(2):
            if time.time() - quiet >= heartbeat:
                print(f"   ... {int(time.time()-t0)}s 仍在运行（{cmd[0]} 无新输出，静默属正常）", flush=True)
                quiet = time.time()
    th = threading.Thread(target=_beat, daemon=True); th.start()
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line)
        if line and not _BAR.match(line) and _HOT.search(line):
            print(f"[{int(time.time()-t0):>3}s] {line}", flush=True)
    rc = proc.wait(); stop.set(); th.join(timeout=1)
    if rc != 0:
        print("---- 命令末尾输出（最多 40 行）----")
        print("\n".join(t for t in tail if t.strip()) or "(无输出)")
    return rc

# conda 装的 kinit 在 sys.prefix/bin：内核重启后 PATH 可能不含它，先补上，
# 否则依赖已齐也会误判 need_kinit，白白再跑一次 conda 求解（本实例实测约 5 分钟）
if os.path.isfile(os.path.join(sys.prefix, "bin", "kinit")):
    os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
need_kinit, need_sasl = shutil.which("kinit") is None, not have("sasl")

# --- 3.1 系统层 ---
if need_kinit or need_sasl:
    apt, env_name = None, "无 root（走 conda 分支）"
    if os.geteuid() == 0:
        apt, env_name = ["apt-get"], "root"
    else:
        sudo_ok = subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0
        if sudo_ok:
            apt, env_name = ["sudo", "-n", "apt-get"], "ma-user + 免密 sudo"
    print(f"[env] {env_name}", flush=True)

    if apt is not None:
        # libsasl2-modules-gssapi-mit / libsasl2-modules = cyrus 的 GSSAPI 插件(必须!)
        if run_stream(apt + ["update"], "apt-get update") != 0:
            raise SystemExit("[FAIL] apt-get update 失败")
        if run_stream(apt + ["install", "-y", "gcc", "g++", "krb5-user",
                             "libkrb5-dev", "libsasl2-dev",
                             "libsasl2-modules-gssapi-mit", "libsasl2-modules"],
                      "apt 安装编译链 + krb5 + cyrus sasl（首次约 1-2 分钟）") != 0:
            raise SystemExit("[FAIL] apt install 失败")
    else:
        # 无 root：ModelArts 自带 anaconda，conda-forge 有预编译的 krb5 和 sasl
        conda = shutil.which("conda")
        assert conda, "[FAIL] 没找到 conda —— 请把报错反馈给维护者"
        print("[env] 无 root —— 走 conda-forge 预编译路径", flush=True)
        # --prefix sys.prefix: 显式装进当前内核环境，避免落到 base
        # --override-channels: 只用本命令指定的频道 —— 实例的 condarc 若配了已失效的
        #   镜像频道(如 TUNA 的 anaconda/pkgs/free, 已停同步 404), 不加这个会一起失败
        base = [conda, "install", "-y", "--override-channels", "--prefix", sys.prefix]
        attempts = [
            (base + ["-c", "conda-forge", "krb5", "sasl"],
             "conda 安装 krb5 + sasl（conda-forge，绕开失效镜像频道；求解+下载 1-3 分钟）"),
            (base + ["-c", "https://conda.anaconda.org/conda-forge", "krb5", "sasl"],
             "conda 重试（官方 conda-forge 源直连，可能较慢）"),
        ]
        rc = 1
        for cmd, note in attempts:
            rc = run_stream(cmd, note)
            if rc == 0:
                break
        if rc != 0:
            raise SystemExit("[FAIL] conda install 两个源均失败（实例镜像源不可用？）；"
                             "备选：改用 OBS 离线 wheel，或把上面的报错反馈给维护者")
        # conda 装的 kinit 在 $CONDA_PREFIX/bin，放进 PATH 供第 5 格使用
        os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
        print("[提示] 若下方 3.3 自检 import 报错（conda 刚装完包内核未感知），"
              "重启内核后重跑第 1、3、4 格即可（已装的会自动跳过）")
else:
    print("[env] 系统依赖已齐（kinit + sasl），跳过安装")

# --- 3.2 python 包（纯 python，pip 即可） ---
PIP_PKGS = [p for p, m in (("pyhive", "pyhive"), ("thrift", "thrift"),
                           ("thrift-sasl", "thrift_sasl"), ("sasl", "sasl"))
            if not have(m)]
if PIP_PKGS:
    if run_stream([sys.executable, "-m", "pip", "install"] + PIP_PKGS,
                  f"pip 安装 {PIP_PKGS}") != 0:
        raise SystemExit("[FAIL] pip install 失败")

# --- 3.3 自检：import + cyrus 的 GSSAPI 插件在位（auth-conf 的硬前提） ---
# 注意: cyrus 的 sasl 包没有"列出机制"的 API（available_mechs 是 pure-sasl 的
# 接口，误用会 AttributeError）。改用功能探测: 真的 init + start 一次 GSSAPI，
# 与第 6 格连接时是同一条代码路径（pyhive.get_sasl_client -> setAttr+init；
# thrift_sasl.open -> start）:
#   start 成功                                  -> 插件在位（且已有票据）
#   报 No worthy mechs / No mechanism available -> 插件缺失（依赖没装全，fatal）
#   报 GSSAPI 凭据类错误（无票据等）           -> 插件在位，第 5 格 kinit 后即可用
import glob
from pyhive import hive
from pyhive.hive import get_installed_sasl
import thrift_sasl
import sasl as cyrus_sasl

_p = cyrus_sasl.Client()
_p.setAttr("host", SPN_HOST)          # 第 1 格的 SPN 中间段，仅作 SASL 层参数
_p.setAttr("service", "hive")
assert _p.init(), f"cyrus sasl 初始化失败: {_p.getError()!r}"
_ok, _mech, _resp = _p.start("GSSAPI")
_err = _p.getError()
_err = _err.decode("utf-8", "replace") if isinstance(_err, bytes) else (_err or "")
if _ok:
    print("[OK] python 依赖就绪；GSSAPI 插件可用；kinit =", shutil.which("kinit"))
elif "worthy mechs" in _err.lower() or "no mechanism available" in _err.lower():
    for _pat in (os.path.join(sys.prefix, "lib*", "sasl2", "*"),
                 "/usr/lib/*/sasl2/*", "/usr/lib64/sasl2/*"):
        for _h in glob.glob(_pat):
            if "gssapi" in os.path.basename(_h).lower():
                print("  gssapi 插件文件:", _h)
    raise SystemExit(
        f"cyrus sasl 缺 GSSAPI 插件（{_err}）\n"
        "root/sudo 环境: 检查 libsasl2-modules-gssapi-mit 是否装上；\n"
        "conda 环境: 把 !ls $CONDA_PREFIX/lib/sasl2/ 的输出发给维护者排查")
else:
    print("[OK] python 依赖就绪；GSSAPI 插件在位（暂无票据，第 5 格 kinit 后生效；"
          f"探测信息: {_err.splitlines()[0] if _err else '-'}）")
    print("kinit =", shutil.which("kinit"))


In [ ]:
# ================== 4. 生成 krb5.conf 并生效 ==================
# dns_canonicalize_hostname=false 是关键：SPN 中间段 hadoop.xxx 是 DNS 里
# 不存在的"假域名"，必须禁止 Kerberos 客户端解析它，原样当 SPN 用。
# udp_preference_limit=1 让 AS/TGS 请求走 TCP —— 与第 2 格探测的 TCP 端口一致。
import os
from pathlib import Path

KRB5_FILE = Path.cwd() / "krb5.conf"
lines = [
    "[libdefaults]",
    f"    default_realm = {REALM}",
    "    dns_canonicalize_hostname = false",   # <- 关键
    "    rdns = false",
    "    udp_preference_limit = 1",
    "",
    "[realms]",
    f"    {REALM} = {{",
    *[f"        kdc = {h}:{KDC_PORT}" for h in KDC_HOSTS],
    f"        admin_server = {KDC_HOSTS[0]}:{KDC_PORT}",
    "    }",
    "",
    "[domain_realm]",
    f"    .{REALM.lower()} = {REALM}",
    f"    {SPN_HOST} = {REALM}",
    f"    .{SPN_HOST} = {REALM}",
    "",
]
KRB5_FILE.write_text("\n".join(lines), encoding="utf-8")
os.environ["KRB5_CONFIG"] = str(KRB5_FILE)   # 后面 kinit 与 cyrus-sasl 都读它
print(f"[OK] 已生成 {KRB5_FILE} 并设置 KRB5_CONFIG\n")
print("\n".join(lines))

In [ ]:
# ================== 5. kinit 获取用户票据（TGT，24h 有效） ==================
# 已有票据则跳过；否则弹出密码输入框（getpass，不在代码里留明文）。
# kinit 可能来自 apt(krb5-user, /usr/bin) 或 conda(krb5, $CONDA_PREFIX/bin)，自适应。
import getpass, shutil, subprocess

KINIT = shutil.which("kinit") or os.path.join(sys.prefix, "bin", "kinit")
KLIST = shutil.which("klist") or os.path.join(sys.prefix, "bin", "klist")

def _run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True,
                          env={**os.environ, "KRB5_CONFIG": str(KRB5_FILE)}, **kw)

r = _run([KLIST])
if r.returncode == 0 and "krbtgt" in r.stdout:
    print("[OK] 已有有效票据，跳过 kinit：")
    print("\n".join(r.stdout.splitlines()[:4]))
else:
    principal = f"{USERNAME}@{REALM}"
    pw = getpass.getpass(f"输入 {principal} 的密码: ")
    r = _run([KINIT, principal], input=pw + "\n")
    assert r.returncode == 0, f"[FAIL] kinit 失败（密码错/KDC 不通？）：{r.stderr.strip()}"
    print(f"[OK] kinit 成功: {principal}")

In [ ]:
# ================== 6. 连接 HiveServer2（核心：解耦 TCP 地址与 SPN） ==================
# pyhive 在 auth=KERBEROS 时把 TCP 连接的 host 直接当 SPN 的 host 用 -> 必然对不上
# （会去 KDC 请求 hive/10.0.0.15@REALM 的票据，而集群注册的是固定串 SPN）。
# 官方逃生口 thrift_transport=...：TCP 层连内网 IP，SASL 层 host 传 SPN 中间段。
# get_installed_sasl 在装了 cyrus 的 sasl 包后自动优先用它（qop=auth-conf 必需）。
from thrift.transport import TSocket

def make_transport():
    tcp = TSocket.TSocket(HIVE_HOST, HIVE_PORT)
    tcp.setTimeout(30000)
    sasl_factory = lambda: get_installed_sasl(
        host=SPN_HOST, sasl_auth="GSSAPI", service="hive")
    return thrift_sasl.TSaslClientTransport(sasl_factory, "GSSAPI", tcp)

PRINCIPAL = f"hive/{SPN_HOST}@{REALM}"
print("尝试 SPN:", PRINCIPAL)
conn = hive.connect(thrift_transport=make_transport(),
                    database=DATABASE, username=USERNAME)
print("[OK] 连接成功！生效 SPN =", PRINCIPAL)

In [ ]:
# ================== 7. 验证查询 ==================
cur = conn.cursor()

cur.execute("SHOW DATABASES")
print("databases:", [r[0] for r in cur.fetchall()])

cur.execute("SHOW TABLES")
tables = [r[0] for r in cur.fetchall()]
print("tables   :", tables)

if "breast_cancer" in tables:
    cur.execute("SELECT COUNT(*) FROM breast_cancer")
    print("breast_cancer 行数:", cur.fetchone()[0])
cur.close()

## 8. 通用查询工具 run_query()

后续任意 SQL 直接 `run_query("...")`。

**排障速查**（完整版见 `hive_export/MRS_RUN.md §5`）：

| 现象 | 处理 |
|---|---|
| 第 3 格 `Permission denied (apt 锁)` | 不是 bug —— 该格自动走 sudo/conda 分支；若仍失败，把 `[env]` 打印和报错反馈维护者 |
| `Invalid token was supplied` | 走了 pure-sasl+pykerberos 路径 → 重跑第 3 格，确认自检打印「GSSAPI 插件在位/可用」，重启内核从头重跑 |
| KDC 探测失败 | 安全组放行 **21732**（TCP+UDP），不是 88 |
| kinit 报密码错 | MRS Manager → 系统 → 用户管理 → 重置 hhx 密码 |
| `import pandas` 报 `numpy.dtype size changed` | 镜像把 `~/modelarts-dev/modelarts-sdk`（含按 numpy 2.x 编译的 pandas 源码树）挂在 sys.path 上，遮蔽了正式安装的 pandas → 下方代码格开头会自动摘除并清理残留；手写 cell 遇到同样报错，重跑本格即可 |

**票据 24h 过期 / 实例重启后**：重跑第 5 格（重新输密码）即可；**重启内核后**第 1、3、4 格必须重跑（PATH/KRB5_CONFIG 等都在内存里）。

**免交互 keytab 备选**：MRS Manager → 系统 → 用户管理 → hhx → 更多 → 下载认证凭据得到 `user.keytab`，上传到 notebook 同目录后，把第 5 格的 kinit 分支替换为：

```python
r = _run([KINIT, "-kt", "user.keytab", f"{USERNAME}@{REALM}"])
```

**用完记得关连接**：`conn.close()`（下方示例末尾已包含）。

In [ ]:
# ================== 8. 通用查询工具 ==================
# 坑（部分 ModelArts 镜像）：~/modelarts-dev/modelarts-sdk 里有一份按 numpy 2.x
# 编译的 pandas 源码树，被挂在 sys.path 上，遮蔽了环境正式安装的 pandas —— 在
# numpy 1.x 内核里 import 直接报 "numpy.dtype size changed"。先摘除再导入。
# 实测（2026-08-19）：本实例有两个遮蔽目录 —— modelarts-sdk、ma-cli 之外还有
# /modelarts/tools/solution/advisor（pandas 2.3.2），摘除前两个后 pandas 从 advisor
# 加载，恰好与内核 numpy 2.0.2 二进制兼容所以可用；靠下方回显的路径确认实际来源。
import subprocess, sys

_dev = [p for p in sys.path if "modelarts-dev" in p or "modelarts-sdk" in p]
for p in _dev:
    sys.path.remove(p)
if _dev:
    print("[fix] 已从 sys.path 摘除开发目录（避免遮蔽正式 pandas）:", ", ".join(_dev))
# 失败的 import 会在 sys.modules 留下半初始化模块，清掉才能就地重试（免重启内核）
for m in [m for m in list(sys.modules) if m == "pandas" or m.startswith("pandas.")]:
    del sys.modules[m]

import numpy as np
try:
    import pandas as pd
except ModuleNotFoundError:                      # 环境里确实没有 pandas 才补装
    subprocess.run([sys.executable, "-m", "pip", "install", "pandas"], check=True)
    import pandas as pd
print(f"[OK] pandas {pd.__version__} <- {pd.__file__}")
print(f"[OK] numpy  {np.__version__} <- {np.__file__}")

def run_query(sql, max_rows=20):
    """执行 SQL，打印并返回 DataFrame（显示截断到 max_rows 行）"""
    cur = conn.cursor()
    try:
        cur.execute(sql)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall()
    finally:
        cur.close()
    df = pd.DataFrame(rows, columns=cols)
    with pd.option_context("display.max_rows", max_rows):
        display(df)
    return df

# 示例：取 5 行看看
_ = run_query("SELECT * FROM breast_cancer LIMIT 5")

# 用完关闭连接：
# conn.close(); print("closed")
